# WHEN NATURE STRIKES
## 01 – Natural Disaster Frequency Analysis

### 1. Introduction

This analysis investigates the frequency and distribution
of different types of natural disasters recorded globally
in the EM-DAT dataset.

The objective is to identify which disaster types occur
most frequently and understand their contribution to
the overall number of recorded natural disaster events.

### 2. Research Questions

1. Which natural disaster type has the highest frequency?
2. Which disaster types contribute most to total events?
3. What percentage of total events does each disaster
   type represent?
4. How concentrated is the distribution among the
   most frequent disaster types?

### 3. Dataset

- Source: EM-DAT (CRED / UCLouvain)
- Main dataset: natural_disaster_events.csv
- Supporting dataset: natural_disaster_type_summary.csv
- Time coverage: 1900–2026
- Total unique events: 15,727

### 4. Methodology

- Use the event-level dataset.
- Group records by Disaster Type.
- Count distinct Event IDs.
- Calculate each disaster type's percentage
  contribution to total events.
- Verify calculated counts against the prepared
  disaster type summary.

### 5. Important Consideration

Each row in the event-level dataset represents one
underlying disaster event. Country-level impact records
are not counted as separate disaster events.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pathlib import Path

sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 7)

print("Libraries imported successfully!")

In [ ]:
# Identify the repository root

ROOT = Path.cwd()

if not (ROOT / "data" / "processed").exists():
    ROOT = ROOT.parent

DATA_DIR = ROOT / "data" / "processed"

# Load datasets

events = pd.read_csv(
    DATA_DIR / "natural_disaster_events.csv",
    encoding="utf-8-sig"
)

type_summary = pd.read_csv(
    DATA_DIR / "natural_disaster_type_summary.csv",
    encoding="utf-8-sig"
)

print("Datasets loaded successfully!")
print("Repository root:", ROOT)

In [ ]:
### 6. Disaster Frequency Calculation

In [ ]:
frequency_df["Cumulative Percentage"] = (
    frequency_df["Percentage"].cumsum()
).round(2)

display(frequency_df)

In [ ]:
verification = frequency_df.merge(
    type_summary[["Disaster Type", "Unique Events"]],
    on="Disaster Type",
    how="outer",
    suffixes=("_Calculated", "_Summary"),
    validate="one_to_one"
)

verification["Match"] = (
    verification["Unique Events_Calculated"]
    == verification["Unique Events_Summary"]
)

display(verification)

print(
    "All disaster frequency counts match:",
    verification["Match"].all()
)

In [ ]:
#Graph 1 — Disaster frequency ranking

In [ ]:
# Create output directory
output_dir = ROOT / "visualizations" / "frequency"
output_dir.mkdir(parents=True, exist_ok=True)

plt.figure(figsize=(13, 8))

ax = sns.barplot(
    data=frequency_df,
    x="Unique Events",
    y="Disaster Type",
    color="steelblue"
)

# Add values beside each bar
for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=5)

plt.title(
    "Frequency of Natural Disasters by Type (1900–2026)",
    fontsize=16,
    fontweight="bold",
    pad=20
)

plt.xlabel("Number of Unique Disaster Events")
plt.ylabel("Disaster Type")

plt.tight_layout()

plt.savefig(
    output_dir / "01_disaster_frequency.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
#Graph 2 — Cumulative frequency distribution

In [ ]:
fig, ax1 = plt.subplots(figsize=(13, 7))

ax1.bar(
    frequency_df["Disaster Type"],
    frequency_df["Unique Events"],
    color="steelblue"
)

ax1.set_xlabel("Disaster Type")
ax1.set_ylabel("Unique Events")
ax1.set_title(
    "Disaster Frequency and Cumulative Distribution",
    fontsize=15,
    fontweight="bold"
)

ax1.tick_params(axis="x", rotation=55)

ax2 = ax1.twinx()

ax2.plot(
    frequency_df["Disaster Type"],
    frequency_df["Cumulative Percentage"],
    marker="o",
    linewidth=2,
    color="darkorange"
)

ax2.set_ylabel("Cumulative Percentage (%)")
ax2.set_ylim(0, 105)

plt.tight_layout()

plt.savefig(
    output_dir / "02_cumulative_frequency.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

In [ ]:
top_5 = frequency_df.head(5)

top_2_percentage = frequency_df.head(2)["Percentage"].sum()

print("TOP 5 MOST FREQUENT DISASTERS")
display(top_5)

print("\nTOTAL UNIQUE EVENTS:", total_events)

print(
    "\nFlood and Storm combined contribution:",
    round(top_2_percentage, 2), "%"
)

print(
    "\nMost frequent disaster:",
    frequency_df.iloc[0]["Disaster Type"]
)

print(
    "Least frequent disaster:",
    frequency_df.iloc[-1]["Disaster Type"]
)